In [34]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [35]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [42]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("../data/kg_train.csv",encoding='latin-1')
# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about...,0
3,Dear Sir=2FMadam=2C I know that this proposal ...,1
4,fyi,0
...,...,...
995,So what's the latest? It sounds contradictory ...,0
996,"TRANSFER OF 36,759,000.00 MILLION POUNDS TO YO...",1
997,Barb I will call to explain. Are you back in t...,0
998,Yang on travelNot free tonite.May work tomorrow,0


### Let's divide the training and test set into two partitions

In [43]:
# Your code
from sklearn.model_selection import train_test_split

data_train, data_val = train_test_split(
    data,
    test_size=0.2,
    random_state=42,
    stratify=data["label"]   # only if you have labels
)


## Data Preprocessing

In [44]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
import nltk
nltk.download('stopwords')
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\swapr\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [45]:
# Your code
import re

def clean_html(raw_html):
    text = raw_html

    # 1. Remove inline JavaScript/CSS
    text = re.sub(r'<script.*?>.*?</script>', '', text, flags=re.DOTALL)
    text = re.sub(r'<style.*?>.*?</style>', '', text, flags=re.DOTALL)

    # 2. Remove HTML comments
    text = re.sub(r'<!--.*?-->', '', text, flags=re.DOTALL)

    # 3. Remove remaining HTML tags
    text = re.sub(r'<[^>]+>', '', text)

    # 4. Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

data["preprocessed_text"] = data["text"].apply(clean_html)



- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [46]:
# Your code
import re

def clean_text_basic(text):

    # 1. Remove all special characters
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)

    # 2. Remove numbers
    text = re.sub(r"\d+", " ", text)

    # 3. Remove all single characters
    text = re.sub(r"\b[a-zA-Z]\b", " ", text)

    # 4. Remove single characters from the start
    text = re.sub(r"^[a-zA-Z]\s+", "", text)

    # 5. Substitute multiple spaces with a single space
    text = re.sub(r"\s+", " ", text)

    # 6. Remove prefixed 'b'
    text = re.sub(r"^b\s+", "", text)

    # 7. Convert to lowercase
    text = text.lower().strip()

    return text

data["preprocessed_text"] = data["text"].apply(clean_text_basic)


## Now let's work on removing stopwords
Remove the stopwords.

In [48]:
# Your code
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

def remove_stopwords(text):
    words = text.split()
    filtered = [w for w in words if w not in stop_words]
    return " ".join(filtered)
data["preprocessed_text"] = data["preprocessed_text"].apply(remove_stopwords)

## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [49]:
# Your code
import nltk
nltk.download("wordnet")
nltk.download("omw-1.4")

#Create the Lemmatizer
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

#Lemmatization Function
def lemmatize_text(text):
    words = text.split()
    lemmas = [lemmatizer.lemmatize(w) for w in words]
    return " ".join(lemmas)

data["preprocessed_text"] = data["preprocessed_text"].apply(lemmatize_text)

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\swapr\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\swapr\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [51]:
# Your code
#split ham and spam
ham = data[data["label"] == 0]["preprocessed_text"]
spam = data[data["label"] == 1]["preprocessed_text"]

In [52]:
#Create Bag‑of‑Words using CountVectorizer
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
ham_bow = vectorizer.fit_transform(ham)
ham_words = vectorizer.get_feature_names_out()

spam_bow = vectorizer.fit_transform(spam)
spam_words = vectorizer.get_feature_names_out()

In [53]:
#Get Top 10 Words in Ham
import numpy as np

ham_counts = np.asarray(ham_bow.sum(axis=0)).flatten()
ham_top10_idx = ham_counts.argsort()[::-1][:10]

print("Top 10 Ham Words:")
for idx in ham_top10_idx:
    print(ham_words[idx], ham_counts[idx])


Top 10 Ham Words:
state 169
pm 127
would 107
president 99
time 95
call 94
mr 91
obama 84
percent 81
secretary 79


In [54]:
#Get Top 10 Words in Spam
spam_counts = np.asarray(spam_bow.sum(axis=0)).flatten()
spam_top10_idx = spam_counts.argsort()[::-1][:10]

print("Top 10 Spam Words:")
for idx in spam_top10_idx:
    print(spam_words[idx], spam_counts[idx])


Top 10 Spam Words:
br 1104
money 987
account 899
bank 801
fund 782
transaction 555
business 514
country 513
mr 490
nbsp 475


## Extra features

In [56]:
from sklearn.model_selection import train_test_split

data_train, data_val = train_test_split(
    data,
    test_size=0.2,
    random_state=42,
    stratify=data["label"]
)

In [57]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['preprocessed_text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['preprocessed_text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['preprocessed_text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['preprocessed_text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['preprocessed_text'].apply(lambda x: len(x)) 

data_train.head()

,text,label,preprocessed_text,money_mark,suspicious_words,text_len
442,Dear=2C Good day hope fine=2Cdear am writting ...,1,dear good day hope fine cdear writting mail du...,1,1,998
962,FROM MR HENRY KABORETHE CHIEF AUDITOR INCHARGE...,1,mr henry kaborethe chief auditor inchargeforei...,0,1,1946
971,Will do.,0,,0,0,0
190,FROM THE DESK OF DR.ADAMU ISMALERAUDITING AND...,1,desk dr adamu ismalerauditing accounting manag...,1,1,383
551,"Dear Friend, My name is LOI C.ESTRADA,The wife...",1,dear friend name loi estrada wife mr josephest...,1,1,1475


## How would you create a Bag of Words with the CountVectorizer method?

In [58]:
# Your code
# Create the vectorizer
vectorizer = CountVectorizer()
# Fit + transform your training text
X_train_bow = vectorizer.fit_transform(data_train["preprocessed_text"])
# Transform validation text
X_val_bow = vectorizer.transform(data_val["preprocessed_text"])
# See the vocabulary (all words)
vectorizer.get_feature_names_out()
X_train_bow.shape

(800, 28304)

## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [59]:
# Your code
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,     
    ngram_range=(1,1),  
    stop_words=None        
)

X_tfidf = tfidf.fit_transform(data["preprocessed_text"])
print(X_tfidf.shape)

(1000, 5000)


### Extra Task (optional) - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

Use a MultinimialNB with default parameters.

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [60]:
# Your code
# MultinomialNB with Bag of Words only

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# Vectorizer
bow = CountVectorizer(max_features=5000)
X_train_bow = bow.fit_transform(data_train["preprocessed_text"])
X_val_bow = bow.transform(data_val["preprocessed_text"])

# Model
clf_bow = MultinomialNB()
clf_bow.fit(X_train_bow, data_train["label"])

# Predict
pred_bow = clf_bow.predict(X_val_bow)

print("Accuracy (Bag of Words):", accuracy_score(data_val["label"], pred_bow))

Accuracy (Bag of Words): 0.98


In [61]:
# MultinomialNB with TF‑IDF only
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(data_train["preprocessed_text"])
X_val_tfidf = tfidf.transform(data_val["preprocessed_text"])

clf_tfidf = MultinomialNB()
clf_tfidf.fit(X_train_tfidf, data_train["label"])

pred_tfidf = clf_tfidf.predict(X_val_tfidf)

print("Accuracy (TF-IDF):", accuracy_score(data_val["label"], pred_tfidf))



Accuracy (TF-IDF): 0.99


In [63]:
# Bag of Words + Engineered Features
import numpy as np

# BOW
X_train_bow = bow.fit_transform(data_train["preprocessed_text"])
X_val_bow = bow.transform(data_val["preprocessed_text"])

# Extra features
train_extra = data_train[["money_mark", "suspicious_words", "text_len"]].values
val_extra = data_val[["money_mark", "suspicious_words", "text_len"]].values

# Combine sparse + dense
from scipy.sparse import hstack
 
X_train_bow_ext = hstack([X_train_bow, train_extra])
X_val_bow_ext = hstack([X_val_bow, val_extra])

clf_bow_ext = MultinomialNB()
clf_bow_ext.fit(X_train_bow_ext, data_train["label"])

pred_bow_ext = clf_bow_ext.predict(X_val_bow_ext)

print("Accuracy (BOW + Extra Features):", accuracy_score(data_val["label"], pred_bow_ext))


Accuracy (BOW + Extra Features): 0.975


In [64]:
# TF‑IDF + Engineered Features
# TF-IDF
X_train_tfidf = tfidf.fit_transform(data_train["preprocessed_text"])
X_val_tfidf = tfidf.transform(data_val["preprocessed_text"])

# Combine with extra features
X_train_tfidf_ext = hstack([X_train_tfidf, train_extra])
X_val_tfidf_ext = hstack([X_val_tfidf, val_extra])

clf_tfidf_ext = MultinomialNB()
clf_tfidf_ext.fit(X_train_tfidf_ext, data_train["label"])

pred_tfidf_ext = clf_tfidf_ext.predict(X_val_tfidf_ext)

print("Accuracy (TF-IDF + Extra Features):", accuracy_score(data_val["label"], pred_tfidf_ext))


Accuracy (TF-IDF + Extra Features): 0.91
